In [9]:
import requests
import pandas as pd

url = "https://fantasy.premierleague.com/api/bootstrap-static/"

response = requests.get(url)
print("Status Code:", response.status_code)
data = response.json()
print(data.keys())

Status Code: 200
dict_keys(['chips', 'events', 'game_settings', 'game_config', 'phases', 'teams', 'total_players', 'element_stats', 'element_types', 'elements'])


### Data Dictionary Notes
- 'elements': have information about a player. It has a lot of attributes (name, team, now_cost, etc...)
- 'teams': have information about teams, such as (name, points, win, etc...)
- 'events': Contain info about each gameweek (most captained, most points, etc...)


### 1.How many players are there?

In [10]:
players = pd.DataFrame(data["elements"])
players.shape

(667, 109)

In [11]:
players[["first_name", "total_points", "now_cost"]].sample(7)

,first_name,total_points,now_cost
9,Benjamin,21,55
86,Junior,0,74
430,Alisson,27,55
84,Alex,2,49
338,Regan,15,45
532,Chido,0,45
569,Matz,14,50


In [12]:
players.columns[50:61]


Index(['minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded',
       'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards',
       'red_cards', 'saves'],
      dtype='str')

### df_players DataFrame
- Identity:  id (unique), code (unique), first_name, second_name. (No null values for all of them)
- Team: team, team_code
- Price: price_change_projections, now_cost
- Form: points_per_game, total_points
- Stats: 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded','own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards',
       'red_cards', 'saves', defensive_contribution

In [13]:
players[["id", "code", "first_name"]].nunique()

id            667
code          667
first_name    494
dtype: int64

- Conclusion: id appears to be the best primary identifier because every player has one and the values are unique.

### 2. How many teams are there?
- What information describes a team?

In [14]:
teams = pd.DataFrame(data["teams"])
teams.shape

(20, 22)

In [15]:
teams.columns

Index(['code', 'draw', 'form', 'id', 'loss', 'name', 'played', 'points',
       'position', 'short_name', 'strength', 'team_division', 'unavailable',
       'win', 'link_url', 'strength_overall_home', 'strength_overall_away',
       'strength_attack_home', 'strength_attack_away', 'strength_defence_home',
       'strength_defence_away', 'pulse_id'],
      dtype='str')

In [37]:
teams.iloc[:2, 10:20]

,strength,team_division,unavailable,win,link_url,strength_overall_home,strength_overall_away,strength_attack_home,strength_attack_away,strength_defence_home
0,None,None,False,0,,4,5,0,0,0
1,None,None,False,0,,3,4,0,0,0


- Identity: id, name, short_name
- Performance: draw, form, loss, played, points, position, win
- Strength: 'strength_overall_home', 'strength_overall_away',
       'strength_attack_home', 'strength_attack_away', 'strength_defence_home',
       'strength_defence_away'
### Observations:
- 'link_url' is a null columns
- 'pulse_id' and 'code' are numbers but no idea what they mean.
- 'unavailable' is all False, need to see what is this
- 'strength' is all None values
- All Performance, strength_* columns: These fields appear to represent dynamic/current-state information and may change as the season progresses.

### Players and Team link

In [31]:
players["team"].isin(teams["id"]).all()

np.True_

In [18]:
mer = pd.merge(players, teams, left_on='team', right_on='id', how='left')
mer["name"].unique()

<StringArray>
[       'Arsenal',    'Aston Villa',    'Bournemouth',      'Brentford',
       'Brighton',        'Chelsea',  'Coventry City', 'Crystal Palace',
        'Everton',         'Fulham',      'Hull City',   'Ipswich Town',
          'Leeds',      'Liverpool',       'Man City',        'Man Utd',
      'Newcastle',  'Nott'm Forest',          'Spurs',     'Sunderland']
Length: 20, dtype: str

- players are linked with teams table via players.team = teams.id

### 3. What information does a gameweek contain?

In [ ]:
gameweek = pd.DataFrame(data["events"])
print(gameweek.shape)
print(gameweek.info())


(38, 29)
<class 'pandas.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 29 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         38 non-null     int64  
 1   name                       38 non-null     str    
 2   deadline_time              38 non-null     str    
 3   release_time               0 non-null      object 
 4   average_entry_score        38 non-null     int64  
 5   finished                   38 non-null     bool   
 6   data_checked               38 non-null     bool   
 7   highest_scoring_entry      5 non-null      float64
 8   deadline_time_epoch        38 non-null     int64  
 9   deadline_time_game_offset  38 non-null     int64  
 10  highest_score              5 non-null      float64
 11  is_previous                38 non-null     bool   
 12  is_current                 38 non-null     bool   
 13  is_next                    38 non-null     bool   
 14

In [73]:
gameweek.columns

Index(['id', 'name', 'deadline_time', 'release_time', 'average_entry_score',
       'finished', 'data_checked', 'highest_scoring_entry',
       'deadline_time_epoch', 'deadline_time_game_offset', 'highest_score',
       'is_previous', 'is_current', 'is_next', 'cup_leagues_created',
       'h2h_ko_matches_created', 'can_enter', 'can_manage', 'released',
       'ranked_count', 'overrides', 'chip_plays', 'most_selected',
       'most_transferred_in', 'top_element', 'top_element_info',
       'transfers_made', 'most_captained', 'most_vice_captained'],
      dtype='str')

In [ ]:
gameweek = pd.DataFrame(data["events"])
gameweek[["is_previous", "is_current", "is_next"]].loc[0:2]

In [ ]:
gameweek[["id", "name", "deadline_time", "released"]].head(3)

In [74]:
gameweek[['most_selected', 'most_transferred_in', 'top_element', 'top_element_info',
       'transfers_made', 'most_captained', 'most_vice_captained']]

,most_selected,most_transferred_in,top_element,top_element_info,transfers_made,most_captained,most_vice_captained
0,411.0,1.0,115.0,"{'id': 115, 'points': 17}",0,411.0,1.0
1,165.0,115.0,426.0,"{'id': 426, 'points': 23}",11562127,411.0,1.0
2,411.0,399.0,204.0,"{'id': 204, 'points': 15}",27412065,411.0,426.0
3,165.0,40.0,124.0,"{'id': 124, 'points': 17}",24092744,411.0,165.0
4,411.0,480.0,552.0,"{'id': 552, 'points': 17}",18907524,411.0,426.0
5,NaN,NaN,NaN,None,4611571,NaN,NaN
6,NaN,NaN,NaN,None,0,NaN,NaN
7,NaN,NaN,NaN,None,0,NaN,NaN
8,NaN,NaN,NaN,None,0,NaN,NaN
9,NaN,NaN,NaN,None,0,NaN,NaN


- Identify: 'id', 'name'
- Time Fields: 'deadline_time', 'release_time' but its all null
- Stats: 'average_entry_score', 'highest_scoring_entry', 'highest_score', 'most_selected', 'most_transferred_in', 'top_element', 'top_element_info',
       'transfers_made', 'most_captained', 'most_vice_captained'
### Observations:
- 38 gameweeks
- 'highest_scoring_entry', 'highest_score': columns that change over time, currently only 5 gameweeks played, so they have 5 non-null values
- 'release_time': all null values, but 'released' has non-null values.
- 'is_*': have all False entries
- 'cup_leagues_created', 'h2h_ko_matches_created': will change over time, especially at the end of the season.

### 4. How does a player connect to a position?

In [20]:
positions = pd.DataFrame(data["element_types"])
print(positions.shape)
positions.head()

(4, 13)


,id,plural_name,plural_name_short,singular_name,singular_name_short,squad_select,squad_min_select,squad_max_select,squad_min_play,squad_max_play,ui_shirt_specific,sub_positions_locked,element_count
0,1,Goalkeepers,GKP,Goalkeeper,GKP,2,None,None,1,1,True,[12],73
1,2,Defenders,DEF,Defender,DEF,5,None,None,3,5,False,[],217
2,3,Midfielders,MID,Midfielder,MID,5,None,None,2,5,False,[],298
3,4,Forwards,FWD,Forward,FWD,3,None,None,1,3,False,[],79


- Only 4 rows (1 for each position)
- 'element_count': sums up to 667 (num of players we got from players df)

In [21]:
print(positions.columns)


Index(['id', 'plural_name', 'plural_name_short', 'singular_name',
       'singular_name_short', 'squad_select', 'squad_min_select',
       'squad_max_select', 'squad_min_play', 'squad_max_play',
       'ui_shirt_specific', 'sub_positions_locked', 'element_count'],
      dtype='str')


In [22]:
mer = pd.merge(players, positions, left_on='element_type', right_on='id', how='left')
players["element_type"].isin(positions["id"]).all()

np.True_

- Conclusion: Players are linked to positions using players.element_type and positions.id. The positions table contains four records representing GK, DEF, MID, and FWD.